# Sentiment Analysis with Scikit-Learn

This notebook builds a sentiment classifier from scratch:
1. Create a labeled dataset of movie review snippets (positive/negative).
2. Build a TF-IDF + Logistic Regression pipeline.
3. Evaluate performance.
4. Visualize the most positive and negative words by model coefficient.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, roc_curve, auc
)

sns.set_style('whitegrid')
print('Imports ready.')

## 1. Create Labeled Dataset

We create a curated dataset of movie review snippets labeled as positive (1) or negative (0).

In [ ]:
positive_reviews = [
    "This movie was absolutely wonderful and I loved every minute of it.",
    "Brilliant performances by the entire cast, truly outstanding film.",
    "A masterpiece of modern cinema, beautifully directed and acted.",
    "I was thoroughly entertained from start to finish. Highly recommended!",
    "The storyline was captivating and the special effects were amazing.",
    "One of the best films I have ever seen. Truly remarkable.",
    "Incredible acting and a beautiful story that touched my heart.",
    "A fantastic movie that exceeded all my expectations.",
    "The director did an excellent job bringing this story to life.",
    "Superb cinematography and a gripping plot throughout.",
    "An uplifting and inspiring film that will stay with you.",
    "The chemistry between the leads was perfect and believable.",
    "A delightful movie with great humor and emotional depth.",
    "Stunning visuals and a soundtrack that perfectly complements the story.",
    "This film is a triumph of storytelling and artistry.",
    "I laughed, I cried, I was completely absorbed in this movie.",
    "The script was clever and the performances were top notch.",
    "A truly enjoyable experience that I would gladly watch again.",
    "The movie had great pacing, wonderful dialogue, and perfect casting.",
    "An outstanding achievement in filmmaking, absolutely loved it.",
    "The plot twists were brilliant and kept me on the edge of my seat.",
    "A feel-good movie with substance and genuine emotion.",
    "Every scene was crafted with care and attention to detail.",
    "The performances were mesmerizing and deeply moving.",
    "A riveting thriller that keeps you guessing until the end.",
    "This is exactly the kind of movie the world needs right now.",
    "Absolutely phenomenal film with incredible depth.",
    "The acting was superb and the story was both original and engaging.",
    "A beautiful film that tells an important story with grace.",
    "I was impressed by the quality of the writing and direction.",
]

negative_reviews = [
    "This movie was terrible and a complete waste of time.",
    "Awful acting and a plot that made absolutely no sense.",
    "I have never seen such a boring and pointless film.",
    "The worst movie I have watched this year, truly dreadful.",
    "Poorly written script with wooden dialogue throughout.",
    "A disaster from start to finish, I wanted to walk out.",
    "The special effects were cheap and the story was predictable.",
    "Disappointing and forgettable, not worth the ticket price.",
    "The pacing was painfully slow and nothing interesting happened.",
    "Terrible direction and an incoherent mess of a story.",
    "I struggled to stay awake during this tedious movie.",
    "The plot holes were enormous and ruined the whole experience.",
    "A sloppy and lazy film that insults the audience.",
    "None of the characters were likeable or even interesting.",
    "The movie dragged on endlessly with no payoff at the end.",
    "Horrible writing and unconvincing performances all around.",
    "This film had no redeeming qualities whatsoever.",
    "A painful viewing experience that I deeply regret.",
    "The dialogue was cringeworthy and the acting was stiff.",
    "An absolute trainwreck of a movie from beginning to end.",
    "Boring, unoriginal, and lacking any real substance.",
    "The movie tried too hard and failed at everything.",
    "Completely overrated and not worth the hype.",
    "A confusing and frustrating film with no clear direction.",
    "I was deeply disappointed by this poorly made movie.",
    "The story was dull and the characters were one dimensional.",
    "A forgettable movie that offers nothing new or exciting.",
    "Uninspired direction and a laughably bad script.",
    "This movie was a chore to sit through, simply awful.",
    "The worst performances I have seen from otherwise talented actors.",
]

texts = positive_reviews + negative_reviews
labels = [1] * len(positive_reviews) + [0] * len(negative_reviews)

df = pd.DataFrame({'text': texts, 'label': labels})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset size: {len(df)}')
print(f'Label distribution:\n{df.label.value_counts()}')
df.head()

## 2. Build TF-IDF + Logistic Regression Pipeline

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.3, random_state=42, stratify=df['label']
)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000, stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000, C=1.0)),
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'\n{classification_report(y_test, y_pred, target_names=["Negative", "Positive"])}')

## 3. Cross-Validation

In [ ]:
cv_scores = cross_val_score(pipeline, df['text'], df['label'], cv=5, scoring='accuracy')
print(f'5-Fold CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
print(f'Per-fold scores: {cv_scores}')

## 4. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn', ax=ax,
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

## 5. Most Positive and Negative Words by Coefficient

Logistic Regression coefficients tell us which words most strongly predict each class.

In [ ]:
# Refit on full dataset for better coefficient analysis
pipeline.fit(df['text'], df['label'])

feature_names = np.array(pipeline.named_steps['tfidf'].get_feature_names_out())
coefficients = pipeline.named_steps['clf'].coef_[0]

# Top positive and negative words
top_n = 20
top_positive_idx = np.argsort(coefficients)[-top_n:]
top_negative_idx = np.argsort(coefficients)[:top_n]

print('Top POSITIVE words:')
for idx in reversed(top_positive_idx):
    print(f'  {feature_names[idx]:25s}  coef: {coefficients[idx]:+.4f}')

print(f'\nTop NEGATIVE words:')
for idx in top_negative_idx:
    print(f'  {feature_names[idx]:25s}  coef: {coefficients[idx]:+.4f}')

In [ ]:
# Visualization: horizontal bar chart of top positive/negative words
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Positive words
pos_words = feature_names[top_positive_idx]
pos_coefs = coefficients[top_positive_idx]
sorted_idx = np.argsort(pos_coefs)
ax1.barh(range(top_n), pos_coefs[sorted_idx], color='#2ecc71', edgecolor='black', linewidth=0.5)
ax1.set_yticks(range(top_n))
ax1.set_yticklabels(pos_words[sorted_idx])
ax1.set_xlabel('Coefficient Weight')
ax1.set_title('Top 20 POSITIVE Sentiment Words', fontsize=13)

# Negative words
neg_words = feature_names[top_negative_idx]
neg_coefs = coefficients[top_negative_idx]
sorted_idx = np.argsort(neg_coefs)[::-1]
ax2.barh(range(top_n), neg_coefs[sorted_idx], color='#e74c3c', edgecolor='black', linewidth=0.5)
ax2.set_yticks(range(top_n))
ax2.set_yticklabels(neg_words[sorted_idx])
ax2.set_xlabel('Coefficient Weight')
ax2.set_title('Top 20 NEGATIVE Sentiment Words', fontsize=13)

plt.suptitle('Words Most Predictive of Sentiment', fontsize=15)
plt.tight_layout()
plt.show()

## 6. Interactive Prediction

In [ ]:
test_sentences = [
    "This movie was absolutely incredible and moving.",
    "What a terrible and boring waste of my evening.",
    "The film was okay, nothing special but not bad either.",
    "Brilliant direction and superb performances throughout.",
    "Awful, dreadful, the worst movie of the decade.",
]

for sentence in test_sentences:
    pred = pipeline.predict([sentence])[0]
    prob = pipeline.predict_proba([sentence])[0]
    sentiment = 'POSITIVE' if pred == 1 else 'NEGATIVE'
    confidence = max(prob)
    print(f'[{sentiment}] (conf: {confidence:.2f}) "{sentence}"')

## Summary

In this notebook we:
1. Created a labeled sentiment dataset of 60 movie review snippets.
2. Built a TF-IDF + Logistic Regression pipeline for binary sentiment classification.
3. Evaluated with cross-validation and confusion matrix.
4. Visualized the most predictive positive and negative words by coefficient weight.
5. Demonstrated interactive sentiment prediction on new sentences.